In [ ]:
!pip install -q transformers accelerate pydantic torch outlines

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.2/102.2 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.4 MB/s eta 0:00:00


## Model Loading

In [ ]:
import torch
import outlines
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

hf_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
hf_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto", torch_dtype=torch.float16
)
model = outlines.from_transformers(hf_model, hf_tokenizer)
tokenizer = hf_tokenizer
print(f"Model loaded on {hf_model.device}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Model loaded on cpu


## Generate with Outlines

In [ ]:
from typing import Type, Optional
from pydantic import BaseModel
import re
from dataclasses import dataclass as dc
@dc
class LLMResponse:
    content: str
    raw: str


def generate(
    messages: list[dict],
    response_format: Optional[Type[BaseModel]] = None,
    max_new_tokens: int = 4096,
    temperature: float = 0.6,
    top_p: float = 0.95,
) -> "LLMResponse | BaseModel":
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    # Structured output — outlines guarantees valid JSON
    if response_format is not None:
        result = model(prompt, response_format, max_new_tokens=max_new_tokens)
        return response_format.model_validate_json(result)

    # Plain text — use transformers directly
    inputs = tokenizer(prompt, return_tensors="pt").to(hf_model.device)
    output_ids = hf_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        do_sample=True,
    )
    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    raw = tokenizer.decode(generated_ids, skip_special_tokens=False)
    content = re.sub(r"<\|[^>]+\|>", "", raw).strip()
    return LLMResponse(content=content, raw=raw)

In [ ]:
# --- Demo: basic generation ---
resp = generate([{"role": "user", "content": "What is 2 + 2?"}])
print("Content:", resp.content)

Content: 2 + 2 = 4.


## Structured Demo

In [ ]:
# --- Demo: structured output with Pydantic + Outlines ---
class MathAnswer(BaseModel):
    reasoning: str
    answer: int

result = generate(
    [{"role": "user", "content": "What is 12 * 13?"}],
    response_format=MathAnswer,
)
print(f"Type:      {type(result).__name__}")
print(f"Reasoning: {result.reasoning}")
print(f"Answer:    {result.answer}")

Type:      MathAnswer
Reasoning: To calculate 12 × 13, we can use the distributive property:

12 × 13 = 12 × (10 + 3) = (12 × 10) + (12 × 3) = 120 + 36 = 156.

Alternatively, you can just multiply directly:

   12
× 13
----
  36  (12 × 3)
120  (12 × 10, shifted one place to the left)
----
156

So, 12 × 13 = 156.
Answer:    156


## Code Executor Demo

In [ ]:
### --- Agent 1: Executor (Python sandbox) ---

import io
import sys
import traceback


class Executor:
    """Executes Python code with a persistent namespace across calls.

    The namespace is shared between runs, so variables defined in step 1
    are available in step 2, etc.  Common libraries are pre-imported.
    """

    def __init__(self):
        self.namespace = {"__builtins__": __builtins__}
        # Pre-import libraries the coder will likely use
        exec(
            "import pandas as pd\nimport numpy as np\n"
            "from scipy import stats\nfrom sklearn import *\n",
            self.namespace,
        )

    def run(self, code: str, timeout: int = 30) -> str:
        """Execute code and return captured stdout/stderr."""
        stdout_buf = io.StringIO()
        stderr_buf = io.StringIO()
        old_stdout, old_stderr = sys.stdout, sys.stderr
        sys.stdout, sys.stderr = stdout_buf, stderr_buf

        try:
            exec(code, self.namespace)
        except Exception:
            traceback.print_exc(file=stderr_buf)
        finally:
            sys.stdout, sys.stderr = old_stdout, old_stderr

        stdout = stdout_buf.getvalue().strip()
        stderr = stderr_buf.getvalue().strip()

        if stderr:
            return f"[STDOUT]\n{stdout}\n\n[ERROR]\n{stderr}" if stdout else f"[ERROR]\n{stderr}"
        return stdout if stdout else "(no output)"

    def reset(self):
        """Clear state for a new task."""
        self.__init__()


# Quick test
exe = Executor()
print(exe.run("x = 40 + 2\nprint(f'The answer is {x}')"))
print(exe.run("print(x * 2)"))  # x persists from previous run

The answer is 42
84
